In [10]:
import os
import json
import yaml
import numpy as np
import librosa
import pandas as pd
import matplotlib.pyplot as plt
import IPython
import soundfile as sf
from tqdm import tqdm 
from collections import OrderedDict
from sklearn.model_selection import train_test_split
import glob

In [11]:
BASE_DIR = "."  


MEDLEYDB_ROOT = os.path.join(BASE_DIR, "MedleyDB")
AUDIO_DIR = os.path.join(MEDLEYDB_ROOT, "Audio")
ANNOTATIONS_DIR = os.path.join(MEDLEYDB_ROOT, "Annotations")


INSTRUMENT_ACTIVATIONS_DIR = os.path.join(ANNOTATIONS_DIR, "Instrument_Activations")
ACTIVATION_CONF_DIR = os.path.join(INSTRUMENT_ACTIVATIONS_DIR, "ACTIVATION_CONF")
SOURCEID_DIR = os.path.join(INSTRUMENT_ACTIVATIONS_DIR, "SOURCEID")

In [12]:
# Definimos primero el mapeo que ya acordamos
# Mapeo de instrumentos originales a categorías agrupadas
instrument_mapping = {
    # Clases individuales grandes
    'electric_bass': 'electric_bass',
    'drum_set': 'drum_set',
    'piano': 'piano',
    'tack_piano': 'piano',
    'synthesizer': 'synthesizer',
    'electric_piano': 'synthesizer',
    'clean_electric_guitar': 'clean_electric_guitar',
    'distorted_electric_guitar': 'distorted_electric_guitar',
    'acoustic_guitar': 'acoustic_guitar',
    'violin': 'violin',
    'female_singer': 'female_singer',
    'vocalists': 'vocalists',
    
    # Male vocals
    'male_singer': 'male_vocals',
    'male_speaker': 'male_vocals',
    'male_rapper': 'male_vocals',
    
    # Otras cuerdas frotadas
    'viola': 'other_bowed_strings',
    'cello': 'other_bowed_strings',
    'double_bass': 'other_bowed_strings',
    'violin_section': 'other_bowed_strings',
    'viola_section': 'other_bowed_strings',
    'cello_section': 'other_bowed_strings',
    'string_section': 'other_bowed_strings',
    'erhu': 'other_bowed_strings',
    
    # Vientos madera
    'flute': 'woodwinds',
    'clarinet': 'woodwinds',
    'oboe': 'woodwinds',
    'bassoon': 'woodwinds',
    'tenor_saxophone': 'woodwinds',
    'soprano_saxophone': 'woodwinds',
    'alto_saxophone': 'woodwinds',
    'baritone_saxophone': 'woodwinds',
    'bass_clarinet': 'woodwinds',
    'piccolo': 'woodwinds',
    'flute_section': 'woodwinds',
    'clarinet_section': 'woodwinds',
    
    # Vientos metal
    'trumpet': 'brass',
    'trombone': 'brass',
    'french_horn': 'brass',
    'french_horn_section': 'brass',
    'brass_section': 'brass',
    'tuba': 'brass',
    'horn_section': 'brass',
    'trumpet_section': 'brass',
    
    # Otra percusión
    'auxiliary_percussion': 'other_percussion',
    'tambourine': 'other_percussion',
    'cymbals': 'other_percussion',
    'timpani': 'other_percussion',
    'tabla': 'other_percussion',
    'shaker': 'other_percussion',
    'snare_drum': 'other_percussion',
    'bass_drum': 'other_percussion',
    'kick_drum': 'other_percussion',
    'bongo': 'other_percussion',
    'darbuka': 'other_percussion',
    'doumbek': 'other_percussion',
    'gong': 'other_percussion',
    'toms': 'other_percussion',
    'drum_machine': 'other_percussion',
    'claps': 'other_percussion',
    'chimes': 'other_percussion',
    
    # Otras cuerdas pulsadas
    'mandolin': 'other_plucked_strings',
    'banjo': 'other_plucked_strings',
    'guzheng': 'other_plucked_strings',
    'oud': 'other_plucked_strings',
    'harp': 'other_plucked_strings',
    'lap_steel_guitar': 'other_plucked_strings',
    'yangqin': 'other_plucked_strings',
    'zhongruan': 'other_plucked_strings',
    'glockenspiel': 'other_plucked_strings',
    'vibraphone': 'other_plucked_strings',
    
    # Instrumentos de lengüetas
    'accordion': 'reed_instruments',
    'harmonica': 'reed_instruments',
    'melodica': 'reed_instruments',
    
    # Efectos y sonidos procesados
    'fx_slash_processed_sound': 'fx_and_processed',
    'scratches': 'fx_and_processed'
}

# Lista de instrumentos a descartar completamente
instruments_to_discard = [
    'sampler',
    'main_system',
    'trombone_section',
]

# Categorías finales
instrument_categories = [
    'electric_bass',
    'drum_set',
    'piano',
    'synthesizer',
    'clean_electric_guitar',
    'distorted_electric_guitar',
    'acoustic_guitar',
    'violin',
    'female_singer',
    'vocalists',
    'male_vocals',
    'other_bowed_strings',
    'woodwinds',
    'brass',
    'other_percussion',
    'other_plucked_strings',
    'reed_instruments',
    'fx_and_processed'
]

# Mapeo de categorías a índices
category_to_idx = {category: idx for idx, category in enumerate(instrument_categories)}

In [13]:
def get_metadata_file_path(track_name):
    """Obtiene la ruta al archivo de metadatos YAML para una pista"""
    return os.path.join(MEDLEYDB_ROOT, "Audio", track_name, f"{track_name}_METADATA.yaml")


def plot_instrument_histogram(instrument_counts, top_n=20, min_count=None):
    """
    Crea un histograma horizontal con las frecuencias de los instrumentos.
    
    Args:
        instrument_counts: Diccionario con instrumento como clave y frecuencia como valor
        top_n: Mostrar solo los top N instrumentos más frecuentes (por defecto 20)
        min_count: Umbral mínimo de apariciones para incluir un instrumento
    """
    # Ordenar instrumentos por frecuencia (mayor a menor)
    sorted_instruments = sorted(instrument_counts.items(), key=lambda x: x[1], reverse=True)
    
    # Filtrar según los parámetros
    filtered_instruments = []
    for instrument, count in sorted_instruments:
        if min_count is not None and count < min_count:
            continue
        filtered_instruments.append((instrument, count))
        if len(filtered_instruments) >= top_n:
            break
    
    # Separar nombres y conteos para el gráfico
    names = [item[0] for item in filtered_instruments]
    counts = [item[1] for item in filtered_instruments]
    
    # Crear figura con tamaño adaptado al número de instrumentos
    plt.figure(figsize=(10, max(8, len(names) * 0.3)))
    
    # Crear barras horizontales
    plt.barh(names, counts, color='skyblue')
    
    # Añadir etiquetas y título
    plt.xlabel('Número de apariciones')
    plt.title('Frecuencia de instrumentos en MedleyDB')
    
    # Ajustar márgenes
    plt.tight_layout()
    
    # Añadir el valor numérico al final de cada barra
    for i, count in enumerate(counts):
        plt.text(count + 0.5, i, str(count), va='center')
    
    # Mostrar el gráfico
    plt.show()

def get_activation_intervals(track_name, instrument):
    """
    Obtiene los intervalos de activación para un instrumento específico.
    
    Args:
        track_name: Nombre del track
        instrument: Nombre del instrumento
        
    Returns:
        list: Lista de tuplas (start_time, end_time)
    """
    sourceid_path = os.path.join(SOURCEID_DIR, f"{track_name}_SOURCEID.lab")
    
    intervals = []
    if os.path.exists(sourceid_path):
        with open(sourceid_path, 'r') as f:
            header = f.readline()  # Saltar encabezado
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3 and parts[2].strip() == instrument:
                    start_time = float(parts[0])
                    end_time = float(parts[1])
                    intervals.append((start_time, end_time))
    
    return intervals

def sanitize_directory_name(name):
    """
    Convierte un nombre de instrumento en un nombre de directorio seguro.
    
    Args:
        name: Nombre original del instrumento
        
    Returns:
        str: Nombre sanitizado seguro para usar como nombre de directorio
    """
    replacements = {
        '/': '_slash_',
        '\\': '_backslash_',
        ':': '_colon_',
        '*': '_star_',
        '?': '_question_',
        '"': '_quote_',
        '<': '_lt_',
        '>': '_gt_',
        '|': '_pipe_',
        ' ': '_'
    }
    
    result = name
    for char, replacement in replacements.items():
        result = result.replace(char, replacement)
    
    return result


def load_audio_segment(filepath, start_time, duration, sr=44100):
    """
    Carga un segmento de audio dado un offset y una duración.
    
    Args:
        filepath: Ruta al archivo .wav
        start_time: Inicio del segmento en segundos
        duration: Duración del segmento en segundos
        sr: Sample rate
        
    Returns:
        np.ndarray: Array de audio
    """
    try:
        y, _ = librosa.load(filepath, sr=sr, offset=start_time, duration=duration)
        return y
    except Exception as e:
        print(f"Error al cargar segmento desde {filepath}: {e}")
        return None


def get_mix_path(track_name):
    """
    Devuelve la ruta al archivo de mezcla (MIX) para un track.
    
    Args:
        track_name: Nombre del track
        
    Returns:
        str: Ruta completa al archivo MIX (.wav)
    """
    return os.path.join(AUDIO_DIR, track_name, f"{track_name}_MIX.wav")


def get_constant_label_intervals(track_name, instrument_list, segment_duration=3.0):
    """
    Encuentra intervalos donde el conjunto de instrumentos activos permanece constante.
    
    Args:
        track_name: Nombre del track
        instrument_list: Lista de instrumentos en orden de STEM
        segment_duration: Duración mínima requerida para aceptar un intervalo
    
    Returns:
        list: Lista de tuplas (start_time, end_time, active_instruments)
    """
    sourceid_path = os.path.join(SOURCEID_DIR, f"{track_name}_SOURCEID.lab")
    if not os.path.exists(sourceid_path):
        print(f"No se encontró SOURCEID para {track_name}")
        return []

    # Paso 1: Cargar todos los eventos
    events = []
    with open(sourceid_path, 'r') as f:
        header = f.readline()
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                start = float(parts[0])
                end = float(parts[1])
                instrument = parts[2].strip()
                if instrument in instrument_list:
                    events.append((start, 'on', instrument))
                    events.append((end, 'off', instrument))
    
    # Paso 2: Ordenar eventos por tiempo
    events.sort()

    # Paso 3: Recorrer eventos y detectar cambios
    intervals = []
    current_time = 0.0
    active_instruments = set()
    
    for event_time, event_type, instrument in events:
        if event_time > current_time and len(active_instruments) > 0:
            duration = event_time - current_time
            if duration >= segment_duration:
                intervals.append((current_time, event_time, set(active_instruments)))
        # Actualizar el estado actual
        if event_type == 'on':
            active_instruments.add(instrument)
        elif event_type == 'off':
            active_instruments.discard(instrument)
        current_time = event_time
    
    return intervals

def get_constant_category_intervals(track_name, segment_duration=3.0):
    """
    Encuentra intervalos donde el conjunto de categorías activas permanece constante.

    Args:
        track_name: Nombre del track
        segment_duration: Duración mínima requerida para aceptar un intervalo

    Returns:
        list: Lista de tuplas (start_time, end_time, active_categories)
    """
    sourceid_path = os.path.join(SOURCEID_DIR, f"{track_name}_SOURCEID.lab")
    if not os.path.exists(sourceid_path):
        print(f"No se encontró SOURCEID para {track_name}")
        return []

    # 1. Leer eventos y mapear a categorías
    raw_events = []
    with open(sourceid_path, 'r') as f:
        f.readline()  # saltar cabecera
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                start = float(parts[0])
                end = float(parts[1])
                instr = sanitize_directory_name(parts[2].strip())
                if instr in instrument_mapping:
                    category = instrument_mapping[instr]
                    raw_events.append((start, 'on', category))
                    raw_events.append((end, 'off', category))
    
    if not raw_events:
        return []

    # 2. Ordenar eventos cronológicamente
    raw_events.sort()

    # 3. Recorrer eventos para formar intervalos
    current_time = 0.0
    active_categories = set()
    intervals = []

    for event_time, event_type, category in raw_events:
        if event_time > current_time and len(active_categories) > 0:
            duration = event_time - current_time
            if duration >= segment_duration:
                intervals.append((current_time, event_time, set(active_categories)))
        if event_type == 'on':
            active_categories.add(category)
        elif event_type == 'off':
            active_categories.discard(category)
        current_time = event_time

    return intervals


def get_stem_instruments(track_name):
    """
    Devuelve una lista de instrumentos en orden de STEM para un track dado.

    Args:
        track_name: Nombre del track

    Returns:
        dict: Diccionario {stem_id: instrumento} ordenado por clave
    """
    metadata_path = get_metadata_file_path(track_name)

    if not os.path.exists(metadata_path):
        print(f"  ⚠️  Metadata no encontrado: {metadata_path}")
        return {}

    with open(metadata_path, 'r') as f:
        metadata = yaml.safe_load(f)

    stem_instruments = {}
    if 'stems' in metadata:
        for stem_id in sorted(metadata['stems'].keys()):  # Ordenar por S01, S02...
            instr = metadata['stems'][stem_id].get('instrument')
            if instr:
                stem_instruments[stem_id] = instr

    return stem_instruments


In [14]:
def extract_multilabel_segments(track_names, segment_duration=3.0, sr=44100, max_segments_per_track=None):
    """
    Extrae segmentos de mezcla con etiquetas multilabel donde los instrumentos activos permanecen constantes.

    Args:
        track_names: Lista de nombres de tracks a procesar
        segment_duration: Duración deseada de cada segmento (en segundos)
        sr: Sample rate para cargar audio
        max_segments_per_track: Máximo número de segmentos a extraer por track (None = sin límite)
    """
    for track_name in track_names:
        print(f"\nProcesando track: {track_name}")
        mix_path = get_mix_path(track_name)

        # Obtener todos los instrumentos presentes según archivos de activación
        stem_instruments = get_stem_instruments(track_name)
        instrument_list = list(stem_instruments.values())

        # Obtener intervalos de etiquetas constantes
        intervals = get_constant_category_intervals(track_name, segment_duration=segment_duration)
        print(f"  → {len(intervals)} intervalos de interés encontrados")
        for i, (start_time, end_time, active_instruments) in enumerate(intervals):
            print(f"    Intervalo {i + 1}: {start_time:.2f}s – {end_time:.2f}s  | Activos: {sorted(active_instruments)}")

        segment_count = 0
        for start_time, end_time, active_instruments in intervals:
            if max_segments_per_track and segment_count >= max_segments_per_track:
                break

            # Convertir a categorías agrupadas
            categories = set(active_instruments)


            if not categories:
                continue  # Skip si no hay clases válidas

            # Vector multi-hot
            label_vector = np.zeros(len(category_to_idx), dtype=np.float32)
            for cat in categories:
                idx = category_to_idx[cat]
                label_vector[idx] = 1.0

            # Extraer audio del segmento
            audio = load_audio_segment(mix_path, start_time, segment_duration, sr=sr)
            if audio is None:
                continue

            # Información útil
            print(f"    Segmento {segment_count + 1}:")
            print(f"      Tiempo: {start_time:.2f}s – {start_time + segment_duration:.2f}s")
            print(f"      Categorías: {sorted(categories)}")
            print(f"      Etiquetas multi-hot: {label_vector.tolist()}")
            print(f"      Duración de audio: {len(audio)/sr:.2f} segundos")

            segment_count += 1


In [15]:
def extract_multilabel_segments(track_names, segment_duration=3.0, sr=44100, max_segments_per_track=None):
    """
    Extrae segmentos de mezcla con etiquetas multilabel donde los instrumentos activos permanecen constantes,
    y guarda los resultados en carpetas individuales además de listas globales.
    """
    os.makedirs("Mix_features", exist_ok=True)
    all_mels, all_cqts, all_labels = [], [], []

    for track_name in track_names:
        print(f"\nProcesando track: {track_name}")
        mix_path = get_mix_path(track_name)

        stem_instruments = get_stem_instruments(track_name)
        instrument_list = list(stem_instruments.values())

        intervals = get_constant_category_intervals(track_name, segment_duration=segment_duration)
        print(f"  → {len(intervals)} intervalos de interés encontrados para {track_name}")
        for i, (start_time, end_time, active_instruments) in enumerate(intervals):
            print(f"    Intervalo {i + 1}: {start_time:.2f}s – {end_time:.2f}s  | Activos: {sorted(active_instruments)}")

        segment_count = 0
        all_segments = []

        # Calcular todos los segmentos posibles en todos los intervalos
        for start_time, end_time, active_instruments in intervals:
            num_possible_segments = int((end_time - start_time) // segment_duration)
            for i in range(num_possible_segments):
                seg_start = start_time + i * segment_duration
                all_segments.append((seg_start, active_instruments))

        for seg_start_time, active_instruments in tqdm(all_segments, desc=f"Extrayendo segmentos de {track_name}"):
            if max_segments_per_track and segment_count >= max_segments_per_track:
                break

            categories = set(active_instruments)
            if not categories:
                continue

            label_vector = np.zeros(len(category_to_idx), dtype=np.float32)
            for cat in categories:
                label_vector[category_to_idx[cat]] = 1.0
            print(label_vector)
            audio = load_audio_segment(mix_path, seg_start_time, segment_duration, sr=sr)
            if audio is None:
                continue

            # Calcular características
            mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=96, hop_length=1024, n_fft=2048)
            mel_db = librosa.power_to_db(mel)
            cqt = librosa.cqt(audio, sr=sr, n_bins=96, hop_length=1024, bins_per_octave=12)
            cqt_mag = np.abs(cqt)

            # Guardar en carpeta individual
            seg_name = f"{track_name}_{segment_count}"
            seg_folder = os.path.join("Mix_features", seg_name)
            os.makedirs(seg_folder, exist_ok=True)

            sf.write(os.path.join(seg_folder, f"{seg_name}.wav"), audio, sr)
            np.save(os.path.join(seg_folder, f"{seg_name}_mel.npy"), mel_db)
            np.save(os.path.join(seg_folder, f"{seg_name}_cqt.npy"), cqt_mag)

            # Acumular globales
            all_mels.append(mel_db)
            all_cqts.append(cqt_mag)
            all_labels.append(label_vector)

            segment_count += 1

    # Guardar listas globales
    np.save("Mix_features/all_mels.npy", np.array(all_mels))
    np.save("Mix_features/all_cqts.npy", np.array(all_cqts))
    np.save("Mix_features/all_labels.npy", np.array(all_labels))


In [16]:
track_names = [name for name in os.listdir(AUDIO_DIR) if os.path.isdir(os.path.join(AUDIO_DIR, name))]
extract_multilabel_segments(track_names, max_segments_per_track=None)



Procesando track: AClassicEducation_NightOwl
  → 12 intervalos de interés encontrados para AClassicEducation_NightOwl
    Intervalo 1: 5.15s – 8.31s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 2: 24.24s – 27.59s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 3: 29.44s – 32.69s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 4: 32.97s – 37.29s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 5: 42.63s – 47.00s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 6: 48.58s – 51.87s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'other_percussion', 'synthesizer'

Extrayendo segmentos de AClassicEducation_NightOwl:  12%|█▏        | 2/17 [00:00<00:00, 17.24it/s]

[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]

Extrayendo segmentos de AClassicEducation_NightOwl:  53%|█████▎    | 9/17 [00:00<00:00, 26.27it/s]


[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1.]
[1. 1. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1.]


...

[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


Extrayendo segmentos de Wolf_DieBekherte:  93%|█████████▎| 50/54 [00:04<00:00, 18.46it/s]

[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


Extrayendo segmentos de Wolf_DieBekherte: 100%|██████████| 54/54 [00:05<00:00, 10.54it/s]

[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [17]:
labels = np.load("Mix_features/all_labels.npy", allow_pickle=True)

In [18]:
labels[0]

array([1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0.,
       0.], dtype=float32)